<a href="https://www.kaggle.com/code/duttaarya/football-transfer-prediction?scriptVersionId=310957772" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🔹Making a forball transfer prediction using on linear regrassion, random forest if needed will update it with another way

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/player-scores/players.csv
/kaggle/input/player-scores/national_teams.csv
/kaggle/input/player-scores/countries.csv
/kaggle/input/player-scores/competitions.csv
/kaggle/input/player-scores/games.csv
/kaggle/input/player-scores/transfers.csv
/kaggle/input/player-scores/game_events.csv
/kaggle/input/player-scores/club_games.csv
/kaggle/input/player-scores/player_valuations.csv
/kaggle/input/player-scores/game_lineups.csv
/kaggle/input/player-scores/appearances.csv
/kaggle/input/player-scores/clubs.csv


In [2]:
players = pd.read_csv("/kaggle/input/player-scores/players.csv")
valuation = pd.read_csv("/kaggle/input/player-scores/player_valuations.csv")
appearances = pd.read_csv("/kaggle/input/player-scores/appearances.csv")

In [3]:
valuation.head()
valuation.columns
valuation.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 616377 entries, 0 to 616376
Data columns (total 6 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   player_id                            616377 non-null  int64  
 1   date                                 616377 non-null  object 
 2   market_value_in_eur                  616377 non-null  int64  
 3   current_club_name                    616377 non-null  object 
 4   current_club_id                      616353 non-null  float64
 5   player_club_domestic_competition_id  544581 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 28.2+ MB


In [4]:
valuation["date"] = pd.to_datetime(valuation["date"])
valuation = valuation.sort_values(["player_id","date"])
latest_valuation = valuation.groupby("player_id").tail(1)

latest_valuation.shape

(39361, 6)

In [5]:
players = pd.read_csv("/kaggle/input/player-scores/players.csv")
players.head()

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0


In [6]:
players = players.drop(columns = [
    "image_url",
    "url",
    "player_code",
    "first_name",
    "last_name",
    "name",
    "city_of_birth",
    "agent_name",
    "highest_market_value_in_eur"
])
#if shows error after using it for multiple time then it will show error as you can't 
#drop the same columns for 2 timws

In [7]:
players.head()
players.shape

(47702, 17)

In [8]:
df = latest_valuation.merge(players, on ="player_id", how = 'left')
df.shape
df.head()

,player_id,date,market_value_in_eur_x,current_club_name_x,current_club_id_x,player_club_domestic_competition_id,last_season,current_club_id_y,country_of_birth,country_of_citizenship,...,position,foot,height_in_cm,contract_expiration_date,international_caps,international_goals,current_national_team_id,current_club_domestic_competition_id,current_club_name_y,market_value_in_eur_y
0,10,2016-01-04,1000000,SS Lazio,398.0,IT1,2015.0,398.0,Poland,Germany,...,Attack,right,184.0,NaN,NaN,NaN,NaN,IT1,Società Sportiva Lazio S.p.A.,1000000.0
1,26,2017-12-28,750000,Borussia Dortmund,16.0,L1,2017.0,16.0,Germany,Germany,...,Goalkeeper,left,190.0,NaN,NaN,NaN,NaN,L1,Borussia Dortmund,750000.0
2,65,2016-06-21,1000000,PAOK Thessaloniki,1091.0,GR1,2015.0,1091.0,Bulgaria,Bulgaria,...,Attack,NaN,NaN,NaN,NaN,NaN,NaN,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0
3,77,2016-11-15,200000,FC Goa,506.0,IT1,2012.0,506.0,Brazil,Brazil,...,Defender,NaN,NaN,NaN,NaN,NaN,NaN,IT1,Juventus Football Club,200000.0
4,80,2018-06-05,100000,Bayern Munich,27.0,L1,2017.0,27.0,East Germany (GDR),Germany,...,Goalkeeper,right,194.0,NaN,NaN,NaN,NaN,L1,FC Bayern München,100000.0


In [9]:
#creating date of birth for age:- 
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"])


In [10]:
df["date_of_birth"].isna().sum()

np.int64(168)

In [11]:
df = df.dropna(subset=["date_of_birth"])
df["date_of_birth"].isna().sum()


np.int64(0)

In [12]:
df["age"] = (pd.to_datetime("today") - df["date_of_birth"]).dt.days / 365
df["age"] = df["age"].astype(int)

In [13]:
df[["date_of_birth", "age"]].head()

,date_of_birth,age
0,1978-06-09,47
1,1980-08-06,45
2,1981-01-30,45
3,1978-05-08,47
4,1981-03-18,45


In [14]:
df = df.drop(columns=["date_of_birth"])

In [15]:
df.columns #checking the name of the column cause after merging the
           # columns they change the name


Index(['player_id', 'date', 'market_value_in_eur_x', 'current_club_name_x',
       'current_club_id_x', 'player_club_domestic_competition_id',
       'last_season', 'current_club_id_y', 'country_of_birth',
       'country_of_citizenship', 'sub_position', 'position', 'foot',
       'height_in_cm', 'contract_expiration_date', 'international_caps',
       'international_goals', 'current_national_team_id',
       'current_club_domestic_competition_id', 'current_club_name_y',
       'market_value_in_eur_y', 'age'],
      dtype='object')

In [16]:
x = df[["age","height_in_cm"]]
y = df["market_value_in_eur_x"]

In [17]:
x.head()
y.head()

0    1000000
1     750000
2    1000000
3     200000
4     100000
Name: market_value_in_eur_x, dtype: int64

In [18]:
x.isna().sum()

age                0
height_in_cm    2014
dtype: int64

In [19]:
df = df.dropna(subset=["age", "height_in_cm"])

In [20]:
x = df[["age","height_in_cm"]]
y = df["market_value_in_eur_x"]

In [21]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [22]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train, y_train)

LinearRegression()

In [23]:
y_pred = model.predict(x_test)

In [24]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)

MAE: 2182102.9105957914
R2 Score: 0.02252581861483094


MAE = Mean Absolute Error

It means:

On average, your model’s prediction is off by ~2.36 million euros.

So if real value = €10M
Model might predict = €7.6M or €12.3M
That is a large error.

this occurs due to low parameters like only height and age

# now taking more 2 parameters (position and foot) 

In [25]:
df["position"].value_counts()


position
Defender      11980
Midfield      10789
Attack        10244
Goalkeeper     4082
Missing          84
Name: count, dtype: int64

In [26]:
df["foot"].value_counts()

foot
right    25112
left      8799
both      1527
Name: count, dtype: int64

In [27]:
df_encoded = pd.get_dummies(df, columns=["position", "foot"], drop_first=True)

In [28]:
df = df.rename(columns={"market_value_in_eur_x": "market_value"})

In [29]:
df = df.rename(columns={"market_value_in_eur_x": "market_value"})
df = df.drop(columns=["market_value_in_eur_y"])

In [30]:
df_encoded = pd.get_dummies(df, columns=["position", "foot"], drop_first=True)

In [31]:
X = df_encoded.drop(columns=["market_value", "player_id"])
y = df_encoded["market_value"]

In [32]:
X = X.select_dtypes(include=["int64", "float64"]) 
#changing the data type for linear regression function

In [33]:
#new added column in 12/04/2026
# Replace inf with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Fill NaN
X = X.fillna(0)

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [35]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [36]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 2091668.277397439
R2: 0.12243512180509142


In [37]:
print("market_value" in X.columns)

False


#   adding another parameter contract length (contract length)

In [38]:
df["contract_expiration_date"] = pd.to_datetime(df["contract_expiration_date"], errors="coerce")

In [39]:
df["contract_years_left"] = (
    df["contract_expiration_date"] - pd.to_datetime("today")
).dt.days / 365

In [40]:
df["contract_years_left"] = df["contract_years_left"].clip(lower=0)

In [41]:
df = df.drop(columns=["contract_expiration_date"])

In [42]:
X.isna().sum()

current_club_id_x           0
last_season                 0
current_club_id_y           0
height_in_cm                0
international_caps          0
international_goals         0
current_national_team_id    0
age                         0
dtype: int64

In [43]:
df_encoded = df_encoded.dropna()

In [44]:
X.isna().sum()

current_club_id_x           0
last_season                 0
current_club_id_y           0
height_in_cm                0
international_caps          0
international_goals         0
current_national_team_id    0
age                         0
dtype: int64

In [45]:
df["contract_years_left"] = df["contract_years_left"].fillna(0)
#filling the missing value with 0

In [46]:
df_encoded = pd.get_dummies(df, columns=["position", "foot"], drop_first=True)

In [47]:
X = df_encoded.drop(columns=["market_value", "player_id"])
y = df_encoded["market_value"]

X = X.select_dtypes(include=["int64", "float64"])

In [48]:
X = X.select_dtypes(include=["int64", "float64"])
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

In [49]:
X.isna().sum()
#checking if there is any another value left undetermined

current_club_id_x           0
last_season                 0
current_club_id_y           0
height_in_cm                0
international_caps          0
international_goals         0
current_national_team_id    0
age                         0
contract_years_left         0
dtype: int64

In [50]:
print("market_value" in X.columns) # checking if there is another 
#lekage is there or not

False


In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [52]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [53]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 1899556.4346374194
R2: 0.21356819353755463


# adding performance features 

In [54]:
appearances = pd.read_csv("/kaggle/input/player-scores/appearances.csv")
appearances.head()

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,76,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90


In [55]:
appearances.columns

Index(['appearance_id', 'game_id', 'player_id', 'player_club_id',
       'player_current_club_id', 'date', 'player_name', 'competition_id',
       'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played'],
      dtype='object')

In [56]:
performance = appearances.groupby("player_id").agg({
    "goals": "sum",
    "assists": "sum",
    "minutes_played": "sum",
    "game_id": "count"
}).reset_index()

performance = performance.rename(columns={
    "game_id": "matches_played"
})

performance.head()
performance.shape

(28665, 5)

In [57]:
df = df.merge(performance, on="player_id", how="left")

In [58]:
df.shape

(37179, 25)

In [59]:
df[["goals", "assists", "minutes_played", "matches_played"]] = \
df[["goals", "assists", "minutes_played", "matches_played"]].fillna(0)

In [60]:
df["goals_per_match"] = df["goals"] / df["matches_played"]
df["assists_per_match"] = df["assists"] / df["matches_played"]

df[["goals_per_match", "assists_per_match"]] = \
df[["goals_per_match", "assists_per_match"]].fillna(0)

In [61]:
df_encoded = pd.get_dummies(df, columns=["position", "foot"], drop_first=True)

In [62]:
features = [
    "age",
    "height_in_cm",
    "contract_years_left",
    "goals_per_match",
    "assists_per_match",
    "matches_played"
]

# Add encoded categorical columns automatically
encoded_cols = [col for col in df_encoded.columns if "position_" in col or "foot_" in col]

X = df_encoded[features + encoded_cols]
y = df_encoded["market_value"]

In [63]:
print("market_value" in X.columns)

False


In [64]:
print(X.isna().sum())

age                    0
height_in_cm           0
contract_years_left    0
goals_per_match        0
assists_per_match      0
matches_played         0
position_Defender      0
position_Goalkeeper    0
position_Midfield      0
position_Missing       0
foot_left              0
foot_right             0
dtype: int64


In [65]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [66]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [67]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 2209782.449534696
R2: 0.23034723937024104


 # 🔹trying random forest way  

In [68]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

from sklearn.metrics import r2_score, mean_absolute_error

print("RF MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RF R2:", r2_score(y_test, y_pred_rf))

RF MAE: 1233632.2435374507
RF R2: 0.5922522509036594


# cheaking for overfitting 

In [69]:
print("Train R2:", rf.score(X_train, y_train))
print("Test R2:", rf.score(X_test, y_test))

Train R2: 0.9398410319375836
Test R2: 0.5922522509036594


# controling the tree depth

In [70]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("Train R2:", rf.score(X_train, y_train))
print("Test R2:", rf.score(X_test, y_test))

Train R2: 0.7530548190493359
Test R2: 0.5690995123691747


# trainging again with random forest 

In [71]:
y_pred_log = rf.predict(X_test)

print("Train R2:", rf.score(X_train, y_train))
print("Test R2:", rf.score(X_test, y_test))

Train R2: 0.7530548190493358
Test R2: 0.5690995123691747


# Using XGBoost 
why using XGboost?
* It perform better for this type of dataset.

In [72]:
!pip install xgboost

In [73]:
from xgboost import XGBRegressor

In [74]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [75]:
xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [76]:
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 1198723.125
R2: 0.624649167060852


Here the R2 is becomes more better than than the previous random forest one by 0.02
